# Предсказание оценки отеля по отзывам

Метрика MAPE

Hotel_Address         -  Адрес отеля, object<br>

Review_Date           -  Дата публикации отзыва, object<br>

Hotel_Name            -  Название отеля, object<br>

Reviewer_Nationality  -  Национальность рецензента, object<br>

Negative_Review       -  Отрицательный отзыв, оставленный рецензентом отелю. В случае отсутствия заполняется значением "No Negative", object<br>

Review_Total_Negative_Word_Counts - Количество слов в отрицательном отзыве, int64<br>

Positive_Review       -  Положительный отзыв, оставленный рецензентом отелю. В случае отсутствия заполняется значением "No Positive", object<br>

Review_Total_Positive_Word_Counts - Количество слов в положительном отзыве, int64<br>

Total_Number_of_Reviews_Reviewer_Has_Given - Количество отзывов, написанных рецензентом в прошлом, int64<br>

Total_Number_of_Reviews - Количество отзывов об отеле, int64<br>

Tags                  -  Теги, данные рецензентом отелю, object<br>

days_since_review     -  Количество дней между написанием отзыва и чисткой, int64<br>

Additional_number_of_soring - Средний балл отеля, на основе всех оценок - с тектом отзыва и без, int64<br>

lat                   -  Широта отеля, float64<br>

lng                   -  Протяженность отеля, float64<br>

**Reviewer_Score**        -  Оценка, данная рецензентом отелю, float64. Целевая переменная

# Загрузка и ознакомление с данными

In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split

import optuna
import xgboost as xgb
from xgboost import XGBRegressor
from sklearn.model_selection import KFold, train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error
from optuna.integration import XGBoostPruningCallback

from dotenv import load_dotenv
load_dotenv()

np.random.seed(42)

In [2]:
train = pd.read_csv(os.getenv('TRAIN_CSV'))          
test = pd.read_csv(os.getenv('TEST_CSV'))

In [3]:
train.head(2)

,Hotel_Address,Additional_Number_of_Scoring,Review_Date,Hotel_Name,Reviewer_Nationality,Negative_Review,Review_Total_Negative_Word_Counts,Total_Number_of_Reviews,Positive_Review,Review_Total_Positive_Word_Counts,Total_Number_of_Reviews_Reviewer_Has_Given,Reviewer_Score,Tags,days_since_review,lat,lng
0,Ndsm Plein 28 Amsterdam Noord 1033 WB Amsterda...,170,4/17/2017,DoubleTree by Hilton Hotel Amsterdam NDSM Wharf,United Kingdom,Too far from attractions Had to use ferry to ...,72,1593,Staff were very helpful Good breakfast,8,1,5.4,"[' Leisure trip ', ' Couple ', ' Queen Guest R...",108 day,52.400181,4.893665
1,Ferdinand Bolstraat 194 Oud Zuid 1072 LW Amste...,114,5/26/2016,Savoy Hotel Amsterdam,Malaysia,Staff should handle customer document during ...,246,995,Very clean and cozy room Friendly and helpful...,41,1,9.6,"[' Leisure trip ', ' Couple ', ' Small Double ...",434 day,52.349743,4.891191


In [4]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 412590 entries, 0 to 412589
Data columns (total 16 columns):
 #   Column                                      Non-Null Count   Dtype  
---  ------                                      --------------   -----  
 0   Hotel_Address                               412590 non-null  object 
 1   Additional_Number_of_Scoring                412590 non-null  int64  
 2   Review_Date                                 412590 non-null  object 
 3   Hotel_Name                                  412590 non-null  object 
 4   Reviewer_Nationality                        412590 non-null  object 
 5   Negative_Review                             412590 non-null  object 
 6   Review_Total_Negative_Word_Counts           412590 non-null  int64  
 7   Total_Number_of_Reviews                     412590 non-null  int64  
 8   Positive_Review                             412590 non-null  object 
 9   Review_Total_Positive_Word_Counts           412590 non-null  int64  
 

In [5]:
test.head(2)

,Hotel_Address,Additional_Number_of_Scoring,Review_Date,Hotel_Name,Reviewer_Nationality,Negative_Review,Review_Total_Negative_Word_Counts,Total_Number_of_Reviews,Positive_Review,Review_Total_Positive_Word_Counts,Total_Number_of_Reviews_Reviewer_Has_Given,Tags,days_since_review,lat,lng
0,7 Western Gateway Royal Victoria Dock Newham L...,359,8/14/2015,Novotel London Excel,United Kingdom,No Negative,0,1158,Excellent location for Excel centre Friendly ...,14,5,"[' Leisure trip ', ' Family with young childre...",720 day,51.507720,0.022981
1,Great Cumberland Place Westminster Borough Lon...,1190,8/3/2017,The Cumberland A Guoman Hotel,Gibraltar,No Negative,0,5180,The location was excellent rieally good next ...,11,2,"[' Leisure trip ', ' Group ', ' Standard Doubl...",0 days,51.514879,-0.160650


In [6]:
test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103148 entries, 0 to 103147
Data columns (total 15 columns):
 #   Column                                      Non-Null Count   Dtype  
---  ------                                      --------------   -----  
 0   Hotel_Address                               103148 non-null  object 
 1   Additional_Number_of_Scoring                103148 non-null  int64  
 2   Review_Date                                 103148 non-null  object 
 3   Hotel_Name                                  103148 non-null  object 
 4   Reviewer_Nationality                        103148 non-null  object 
 5   Negative_Review                             103148 non-null  object 
 6   Review_Total_Negative_Word_Counts           103148 non-null  int64  
 7   Total_Number_of_Reviews                     103148 non-null  int64  
 8   Positive_Review                             103148 non-null  object 
 9   Review_Total_Positive_Word_Counts           103148 non-null  int64  
 

In [7]:
# Проверка на пропуски в данных
display(train.isnull().sum())
test.isnull().sum()

Hotel_Address                                    0
Additional_Number_of_Scoring                     0
Review_Date                                      0
Hotel_Name                                       0
Reviewer_Nationality                             0
Negative_Review                                  0
Review_Total_Negative_Word_Counts                0
Total_Number_of_Reviews                          0
Positive_Review                                  0
Review_Total_Positive_Word_Counts                0
Total_Number_of_Reviews_Reviewer_Has_Given       0
Reviewer_Score                                   0
Tags                                             0
days_since_review                                0
lat                                           2600
lng                                           2600
dtype: int64

Hotel_Address                                   0
Additional_Number_of_Scoring                    0
Review_Date                                     0
Hotel_Name                                      0
Reviewer_Nationality                            0
Negative_Review                                 0
Review_Total_Negative_Word_Counts               0
Total_Number_of_Reviews                         0
Positive_Review                                 0
Review_Total_Positive_Word_Counts               0
Total_Number_of_Reviews_Reviewer_Has_Given      0
Tags                                            0
days_since_review                               0
lat                                           668
lng                                           668
dtype: int64

In [8]:
# Удаление пропусков и проверка

train = train.dropna()
test = test.dropna()
display(train.isnull().sum())
test.isnull().sum()

Hotel_Address                                 0
Additional_Number_of_Scoring                  0
Review_Date                                   0
Hotel_Name                                    0
Reviewer_Nationality                          0
Negative_Review                               0
Review_Total_Negative_Word_Counts             0
Total_Number_of_Reviews                       0
Positive_Review                               0
Review_Total_Positive_Word_Counts             0
Total_Number_of_Reviews_Reviewer_Has_Given    0
Reviewer_Score                                0
Tags                                          0
days_since_review                             0
lat                                           0
lng                                           0
dtype: int64

Hotel_Address                                 0
Additional_Number_of_Scoring                  0
Review_Date                                   0
Hotel_Name                                    0
Reviewer_Nationality                          0
Negative_Review                               0
Review_Total_Negative_Word_Counts             0
Total_Number_of_Reviews                       0
Positive_Review                               0
Review_Total_Positive_Word_Counts             0
Total_Number_of_Reviews_Reviewer_Has_Given    0
Tags                                          0
days_since_review                             0
lat                                           0
lng                                           0
dtype: int64

## Подготовка признаков

In [9]:
# Преобразование столбца 'days_since_review' в числовой формат
train['days_since_review'] = train['days_since_review'].str.split(' ').str[0].astype(int)
test['days_since_review'] = test['days_since_review'].str.split(' ').str[0].astype(int)
train['days_since_review'].head(2)


0    108
1    434
Name: days_since_review, dtype: int64

In [10]:
# Преобразование столбца 'review_date' в datetime формат
# Преобразование и извлечение признаков — надёжно для train и test
for df in (train, test):
    # убедимся, что колонка есть и преобразуем в datetime
    if 'Review_Date' in df.columns:
        df['Review_Date'] = pd.to_datetime(df['Review_Date'], errors='coerce')
        df['review_year']  = df['Review_Date'].dt.year
        df['review_month'] = df['Review_Date'].dt.month
        df['review_day']   = df['Review_Date'].dt.day
    else:
        raise KeyError("В DataFrame нет колонки 'Review_Date'")

# корректный вывод — имя колонки с правильным регистром
display_cols = ['Review_Date', 'review_year', 'review_month', 'review_day']
train[display_cols].head(2)
train.drop(columns=['Review_Date'], inplace=True)
test.drop(columns=['Review_Date'], inplace=True)

In [11]:
Reviewer_Nationality_unique = train['Reviewer_Nationality'].unique().tolist()
Reviewer_Nationality_unique

[' United Kingdom ',
 ' Malaysia ',
 ' Oman ',
 ' Netherlands ',
 ' Kuwait ',
 ' New Zealand ',
 ' United States of America ',
 ' Botswana ',
 ' Germany ',
 ' Czech Republic ',
 ' Jordan ',
 ' Canada ',
 ' Ireland ',
 ' United Arab Emirates ',
 ' Latvia ',
 ' Italy ',
 ' Australia ',
 ' South Korea ',
 ' Portugal ',
 ' Switzerland ',
 ' Hong Kong ',
 ' Bahamas ',
 ' Sweden ',
 ' Spain ',
 ' Romania ',
 ' Egypt ',
 ' India ',
 ' Turkey ',
 ' Nigeria ',
 ' Greece ',
 ' Andorra ',
 ' Iran ',
 ' Estonia ',
 ' Denmark ',
 ' France ',
 ' Saudi Arabia ',
 ' Thailand ',
 ' Pakistan ',
 ' Singapore ',
 ' South Africa ',
 ' Belgium ',
 ' Bermuda ',
 ' Bahrain ',
 ' Israel ',
 ' Cyprus ',
 ' China ',
 ' Taiwan ',
 ' Russia ',
 ' Lebanon ',
 ' Lithuania ',
 ' Norway ',
 ' Poland ',
 ' Indonesia ',
 ' Mexico ',
 ' Hungary ',
 ' Peru ',
 ' Philippines ',
 ' Serbia ',
 ' Finland ',
 ' Colombia ',
 ' Croatia ',
 ' Guernsey ',
 ' Bulgaria ',
 ' Malta ',
 ' Luxembourg ',
 ' Venezuela ',
 ' Brazil ',
 ' 

## Обучение модели

In [12]:
# X = train.drop('Reviewer_Score', axis=1)
# y = train['Reviewer_Score']
X = train[['Additional_Number_of_Scoring', 'Review_Total_Negative_Word_Counts',
           'Review_Total_Positive_Word_Counts','Total_Number_of_Reviews', 'Total_Number_of_Reviews_Reviewer_Has_Given',
           'lat', 'lng', 'days_since_review', 'Reviewer_Score', 'review_year', 'review_month', 'review_day']].drop(columns='Reviewer_Score')
y = train['Reviewer_Score']

# Разделяем на X_train, X_temp, y_train, y_temp для последующего выделения валидационного набора
X = X.values
y = y.values

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=42)

# Разделяем временные данные на валидационные и тестовые
X_valid, X_test, y_valid, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

In [13]:
# === Настройка кросс-валидации ===
kf = KFold(n_splits=3, shuffle=True, random_state=42)

def safe_mape(y_true: np.ndarray, y_pred: np.ndarray, eps: float = 1e-6) -> float:
    """Безопасный MAPE для избежания деления на ноль."""
    denom = np.maximum(np.abs(y_true), eps)
    return np.mean(np.abs((y_true - y_pred) / denom)) * 100

# === Целевая функция для Optuna ===
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.3, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 20),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 5.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 5.0),
        'random_state': 42,
        'n_jobs': -1,
        'verbosity': 0
    }

    cv_scores = []
    # Используем только тренировочные данные для кросс-валидации
    for fold, (train_idx, val_idx) in enumerate(kf.split(X_train)):
        X_tr, X_val = X_train[train_idx], X_train[val_idx]
        y_tr, y_val = y_train[train_idx], y_train[val_idx]

        model = XGBRegressor(**params)
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            verbose=False
        )

        preds = model.predict(X_val)
        score = safe_mape(y_val, preds)
        cv_scores.append(score)

        # Отчет для возможности pruning
        trial.report(score, fold)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return float(np.mean(cv_scores))

# === Оптимизация гиперпараметров ===
study = optuna.create_study(
    direction='minimize',
    sampler=optuna.samplers.TPESampler(seed=42)
)
study.optimize(objective, n_trials=20)

print("✅ Best params:", study.best_params)
print("✅ Best CV MAPE:", study.best_value)

# === Обучение финальной модели ===
best_params = study.best_params.copy()
best_params.update({
    'n_jobs': -1, 
    'random_state': 42, 
    'verbosity': 0
})

# Обучаем на объединенных тренировочных и валидационных данных для лучшего качества
X_train_valid = np.vstack([X_train, X_valid])
y_train_valid = np.concatenate([y_train, y_valid])

final_model = XGBRegressor(**best_params)
final_model.fit(X_train_valid, y_train_valid, verbose=False)

# === Предсказания на тестовом наборе ===
test_preds = final_model.predict(X_test)
test_mape = safe_mape(y_test, test_preds)

print("✅ Test MAPE:", test_mape)

[I 2025-11-05 19:11:13,270] A new study created in memory with name: no-name-9df00657-22cb-4bdd-b5f6-cef0cfb95d9b
[I 2025-11-05 19:11:18,230] Trial 0 finished with value: 14.365907882366548 and parameters: {'n_estimators': 437, 'learning_rate': 0.22648248189516848, 'max_depth': 10, 'subsample': 0.7993292420985183, 'colsample_bytree': 0.4936111842654619, 'min_child_weight': 4, 'reg_alpha': 0.2904180608409973, 'reg_lambda': 4.330880728874676}. Best is trial 0 with value: 14.365907882366548.
[I 2025-11-05 19:11:20,872] Trial 1 finished with value: 13.758443175068068 and parameters: {'n_estimators': 641, 'learning_rate': 0.05675206026988748, 'max_depth': 3, 'subsample': 0.9849549260809971, 'colsample_bytree': 0.899465584480253, 'min_child_weight': 5, 'reg_alpha': 0.9091248360355031, 'reg_lambda': 0.9170225492671691}. Best is trial 1 with value: 13.758443175068068.
[I 2025-11-05 19:11:23,620] Trial 2 finished with value: 13.747754019635769 and parameters: {'n_estimators': 374, 'learning_rat

✅ Best params: {'n_estimators': 497, 'learning_rate': 0.10540271264874637, 'max_depth': 5, 'subsample': 0.8264648750204893, 'colsample_bytree': 0.6829554234304778, 'min_child_weight': 6, 'reg_alpha': 2.7423828756162116, 'reg_lambda': 1.0646136607682553}
✅ Best CV MAPE: 13.547817236629998
✅ Test MAPE: 13.477477701055529
